In [1]:
from ultralytics import YOLO
import cv2

### Checking CUDA

In [6]:
import torch
import subprocess
import sys

def check_cuda():
    print(f"Python Version: {sys.version}")
    print("\n--- Checking PyTorch CUDA ---")
    try:
        print(f"PyTorch Version: {torch.__version__}")
        if torch.cuda.is_available():
            print(f"CUDA is available! Device count: {torch.cuda.device_count()}")
            print(f"Current device: {torch.cuda.get_device_name(0)}")
        else:
            print("CUDA is NOT available in PyTorch.")
    except ImportError:
        print("PyTorch is not installed.")

    print("\n--- Checking nvidia-smi ---")
    try:
        result = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if result.returncode == 0:
            print("nvidia-smi found.")
            print(result.stdout[:500])
        else:
            print("nvidia-smi error.")
    except FileNotFoundError:
        print("nvidia-smi command not found.")

check_cuda()

Python Version: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]

--- Checking PyTorch CUDA ---
PyTorch Version: 2.9.1+cu130
CUDA is available! Device count: 1
Current device: NVIDIA GeForce RTX 3050 Laptop GPU

--- Checking nvidia-smi ---
nvidia-smi found.
Mon Dec  8 09:41:24 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.42                 Driver Version: 581.42         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|       


### Choosing Model

In [ ]:
base_model = YOLO('yolov8n.pt')
print("Current model is", base_model.model_name)

Current model is yolov8n.pt


In [8]:
from custom.custom_yolo import YOLO_FocalEIoU

model = YOLO_FocalEIoU("yolov8n.yaml")

### Train

In [15]:
results = model.train(
    data="dataset/v1/data.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    workers=8,
    seed=14,
    optimizer="auto",
    patience=50,
    save_period=20,
    val=True,
    device=0,
    plots=True,
    exist_ok=True,
    project="runs/train",
    name="somba-yolov8-train",
    verbose=True,
)

New https://pypi.org/project/ultralytics/8.3.235 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.234  Python-3.13.7 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)


AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
img = cv2.imread('/runs/train/somba-yolov8-train/results.png')
plt.figure(figsize=(12,8))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

### Validation

In [ ]:
metrics = model.val(
    data='dataset/v1/data.yaml',
    imgsz=1088,
    batch=32,
    iou=0.7,
    workers=8,
    device="cuda:0",
    plots=True,
    project="runs/val",
    verbose=True,
    name="somba-yolov8-val",
    visualize=True,
)

In [ ]:
print(results.to_df())

In [ ]:
test = model.test()